<a href="https://colab.research.google.com/github/mkvkanpur/hpc/blob/main/struct_warp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
try:
    import warp as wp
    print(f"Warp version {wp.__version__} is ready!")
except ImportError:
    print("Warp not found. Installing...")
    !pip install warp-lang
    import warp as wp
    wp.init()

# Check if CUDA is actually available on this specific machine
if wp.is_cuda_available():
    device = "cuda"
else:
    device = "cpu"
    print("Warning: CUDA not found. Falling back to CPU.")

print(f"Using device: {device}")

Warp version 1.12.0 is ready!
Using device: cuda


# 2D Hydro

In [63]:
import warp as wp
import torch
import numpy as np

wp.init()

N = 64
Nx = N
Ny = N
STR_SIZE = int(np.sqrt(Nx**2 + Ny**2)/2)
TILE_x = wp.constant(8)
TILE_y = wp.constant(8)
NO_TILES_x = (Nx//(2*TILE_x))
NO_TILES_y = (Ny//(2*TILE_y))


@wp.kernel
def compute_str(
    ux: wp.array2d(dtype=float),
    uy: wp.array2d(dtype=float),
    str_tiled: wp.array3d(dtype=float),
    l_min: wp.vec2, l_max: wp.vec2
):
    i, j = wp.tid()

    ux_tile = wp.tile_load(ux, shape=(TILE_x, TILE_y), offset=(i*TILE_x, j*TILE_y))
    uy_tile = wp.tile_load(uy, shape=(TILE_x, TILE_y), offset=(i*TILE_x, j*TILE_y))

    for lx in range(int(l_min[0]), int(l_max[0])):
      for ly in range(int(l_min[1]), int(l_max[1])):

        ux_tile_shifted = wp.tile_load(ux, shape=(TILE_x, TILE_y), offset=(i*TILE_x+lx, j*TILE_y+ly))
        uy_tile_shifted = wp.tile_load(uy, shape=(TILE_x, TILE_y), offset=(i*TILE_x+lx, j*TILE_y+ly))

        dux_tile = ux_tile_shifted - ux_tile
        duy_tile = uy_tile_shifted - uy_tile

        l_vec = wp.vec2(float(lx), float(ly))
        l_mag = wp.length(l_vec)

        # 2. Now the math will work because these are tiles of floats
        #l_mag = wp.length(wp.vec2(float(lx), float(ly)))

        # Calculate the base projection value for each element in the tile
        base_projection_tile = (dux_tile * float(lx) + duy_tile * float(ly))/l_mag

        # for x in range(TILE_x):
        #     for y in range(TILE_y):
        #         base_projection_tile[x, y] = wp.pow(base_projection_tile[x, y], 3.0)

        base_projection_tile = base_projection_tile * base_projection_tile * base_projection_tile

        tile_sum = wp.tile_sum(base_projection_tile)

        num_elements = float(TILE_x * TILE_y)

        partial_mean = tile_sum[0]/ num_elements
        str_index = int(l_mag)
        str_tiled[str_index, i, j] = partial_mean


### MAIN ####
ux_np = np.arange(N, dtype=np.float32).reshape(-1, 1) * np.ones((1, N), dtype=np.float32)
uy_np = np.arange(N, dtype=np.float32).reshape(1, -1) * np.ones((N, 1), dtype=np.float32)

print(ux_np[2,2], uy_np[2,2])

ux_wp = wp.array(ux_np, dtype=float, device="cuda")
uy_wp = wp.array(uy_np, dtype=float, device="cuda")

str_tiled = wp.zeros((STR_SIZE,NO_TILES_x, NO_TILES_y), dtype=float)
str = wp.zeros(STR_SIZE, dtype=float)

l_min = wp.vec2(2.0, 2.0)
l_max = wp.vec2(float(Nx//3), float(Ny//3))
# # Corrected block_dim from TILE_THREADS to (TILE_x, TILE_y)
wp.launch_tiled(compute_str, dim=[NO_TILES_x, NO_TILES_y], inputs=[ux_wp, uy_wp, str_tiled, l_min, l_max], block_dim=128)
wp.synchronize()

# Convert wp.array to torch.Tensor before calling torch.sum
str_tiled_torch = wp.to_torch(str_tiled)
str = torch.sum(str_tiled_torch, dim=(1, 2))/(NO_TILES_x*NO_TILES_y)
print(f"str = {str}")

2.0 2.0
Module __main__ e824094 load on device 'cuda:0' took 3.60 ms  (cached)
str = tensor([0.0000e+00, 0.0000e+00, 2.2627e+01, 4.6872e+01, 8.9443e+01, 1.9825e+02,
        3.0187e+02, 4.4171e+02, 7.1554e+02, 9.5534e+02, 1.2494e+03, 1.6035e+03,
        2.0239e+03, 2.7021e+03, 3.2854e+03, 3.9528e+03, 4.7104e+03, 5.5641e+03,
        6.8305e+03, 7.9102e+03, 9.1039e+03, 1.0549e+04, 1.1892e+04, 1.3573e+04,
        1.4550e+04, 1.6802e+04, 1.9481e+04, 2.0993e+04, 2.2627e+04, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00], device='cuda:0')


# 3D Hydro: S3_Kolm

In [76]:
import warp as wp
import torch
import numpy as np

wp.init()

N = 64
Nx = N
Ny = N
Nz = N
STR_SIZE = int(np.sqrt(Nx**2 + Ny**2 + Nz**2)/2)
TILE_x = wp.constant(8)
TILE_y = wp.constant(8)
TILE_z = wp.constant(8)
NO_TILES_x = (Nx//(2*TILE_x))
NO_TILES_y = (Ny//(2*TILE_y))
NO_TILES_z = (Nz//(2*TILE_z))


@wp.kernel
def compute_str(
    ux: wp.array3d(dtype=float),
    uy: wp.array3d(dtype=float),
    uz: wp.array3d(dtype=float),
    str_tiled: wp.array4d(dtype=float), # Changed from wp.array3d to wp.array4d
    l_min: wp.vec3, l_max: wp.vec3
):
    i, j, k = wp.tid()

    ux_tile = wp.tile_load(ux, shape=(TILE_x, TILE_y, TILE_z), offset=(i*TILE_x, j*TILE_y, k*TILE_z))
    uy_tile = wp.tile_load(uy, shape=(TILE_x, TILE_y, TILE_z), offset=(i*TILE_x, j*TILE_y, k*TILE_z))
    uz_tile = wp.tile_load(uz, shape=(TILE_x, TILE_y, TILE_z), offset=(i*TILE_x, j*TILE_y, k*TILE_z))

    for lx in range(int(l_min[0]), int(l_max[0])):
      for ly in range(int(l_min[1]), int(l_max[1])):
        for lz in range(int(l_min[2]), int(l_max[2])):

          ux_tile_shifted = wp.tile_load(ux, shape=(TILE_x, TILE_y, TILE_z), offset=(i*TILE_x+lx, j*TILE_y+ly, k*TILE_z+lz))
          uy_tile_shifted = wp.tile_load(uy, shape=(TILE_x, TILE_y, TILE_z), offset=(i*TILE_x+lx, j*TILE_y+ly, k*TILE_z+lz))
          uz_tile_shifted = wp.tile_load(uz, shape=(TILE_x, TILE_y, TILE_z), offset=(i*TILE_x+lx, j*TILE_y+ly, k*TILE_z+lz))

          dux_tile = ux_tile_shifted - ux_tile
          duy_tile = uy_tile_shifted - uy_tile
          duz_tile = uz_tile_shifted - uz_tile

          l_vec = wp.vec3(float(lx), float(ly), float(lz))
          l_mag = wp.length(l_vec)

          base_projection_tile = (dux_tile * float(lx) + duy_tile * float(ly) + duz_tile * float(lz))/l_mag

          base_projection_tile = base_projection_tile * base_projection_tile * base_projection_tile

          tile_sum = wp.tile_sum(base_projection_tile)

          num_elements = float(TILE_x * TILE_y * TILE_z)

          partial_mean = tile_sum[0]/ num_elements
          str_index = int(l_mag)
          str_tiled[str_index, i, j, k] = partial_mean


### MAIN ####
ux_np = np.arange(N, dtype=np.float32).reshape(-1, 1, 1) * np.ones((1, N, N), dtype=np.float32)
uy_np = np.arange(N, dtype=np.float32).reshape(1, -1, 1) * np.ones((N, 1, N), dtype=np.float32)
uz_np = np.arange(N, dtype=np.float32).reshape(1, 1, -1) * np.ones((N, N, 1), dtype=np.float32)

print(ux_np[2,2,2], uy_np[2,2,2])

ux_wp = wp.array(ux_np, dtype=float, device="cuda")
uy_wp = wp.array(uy_np, dtype=float, device="cuda")
uz_wp = wp.array(uz_np, dtype=float, device="cuda")


str_tiled = wp.zeros((STR_SIZE, NO_TILES_x, NO_TILES_y, NO_TILES_z), dtype=float)
str = wp.zeros(STR_SIZE, dtype=float)

l_min = wp.vec3(2.0, 2.0, 2.0)
l_max = wp.vec3(float(Nx//3), float(Ny//3), float(Nz//3))
# # Corrected block_dim from TILE_THREADS to (TILE_x, TILE_y)
wp.launch_tiled(compute_str, dim=[NO_TILES_x, NO_TILES_y, NO_TILES_z], inputs=[ux_wp, uy_wp, uz_wp, str_tiled, l_min, l_max], block_dim=128)
wp.synchronize()

# Convert wp.array to torch.Tensor before calling torch.sum
str_tiled_torch = wp.to_torch(str_tiled)
str = torch.sum(str_tiled_torch, dim=(1, 2, 3))/(NO_TILES_x * NO_TILES_y * NO_TILES_z)
print(f"str = {str}")

2.0 2.0
Module __main__ 376bae9 load on device 'cuda:0' took 3.61 ms  (cached)
str = tensor([    0.0000,     0.0000,     0.0000,    41.5692,   117.5755,   189.5706,
          291.8629,   488.1885,   675.6722,   985.0377,  1314.5339,  1674.2822,
         2100.2249,  2702.1069,  3285.4011,  4048.0940,  4811.3535,  5805.0210,
         6773.6792,  7850.4702,  9229.5176, 10516.2715, 12029.2607, 13716.1406,
        15475.2383, 17420.2344, 19642.5117, 21658.6562, 24171.8262, 26149.5293,
        29004.0059, 31433.2285, 34315.9297, 37683.3555, 41569.2109,     0.0000,
            0.0000,     0.0000,     0.0000,     0.0000,     0.0000,     0.0000,
            0.0000,     0.0000,     0.0000,     0.0000,     0.0000,     0.0000,
            0.0000,     0.0000,     0.0000,     0.0000,     0.0000,     0.0000,
            0.0000], device='cuda:0')


In [ ]:
N = 10

ux = np.arange(N, dtype=np.float32).reshape(-1, 1) * np.ones((1, N), dtype=np.float32)
uy = np.arange(N, dtype=np.float32).reshape(1, -1) * np.ones((N, 1), dtype=np.float32)
str = np.zeros(N, dtype=np.float32)

ux_t = torch.from_numpy(ux).to(device)
uy_t = torch.from_numpy(uy).to(device)
str_t = torch.from_numpy(str).to(device)

# 2. Combine into a vec2 field and "Wrap" it
# Shape: (limit, limit, 2)
u_combined = torch.stack([ux_t, uy_t], dim=-1).contiguous()
u_wp = wp.from_torch(u_combined, dtype=wp.vec2)

l_min = [1,1]
l_max = [4,4]

up_wp = u_wp[0:3, 0:2]
print(up_wp)

[[[0. 0.]
  [0. 1.]]

 [[1. 0.]
  [1. 1.]]

 [[2. 0.]
  [2. 1.]]]
